# N/V 조건부 ID 임베딩 변환 M2 — Dunnhumby seed 42

N/V가 별도 상품점수 공간을 만들지 않고, 기존 64차원 사용자 ID 선호 임베딩을 제한적으로 변환하는 구조를 빠르게 확인합니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 사용자 표현: `Norm(E_u + 0.05[c_N(u)A_N E_u + c_V(u)A_V E_u])`
- N/V 변환: 각각 rank 4
- 아이템 표현: 순수 64차원 ID 임베딩(경제·인기도 특징 없음)
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch
- 비교: 동일 protocol·입력 manifest의 기존 M1@64 결과 재사용

이 실행은 역사적 개발구간 seed 42 탐색이며 유의성을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '93790846cd4d9f9832373b7b7849291177c6b4e4'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_conditional_id_transform import (
    configure_conditional_id_transform_run,
    preflight_summary,
    run_conditional_id_transform_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_conditional_id_transform_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_conditional_id_transform_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['m2']['embedding_dim'] == 64
assert summary['m2']['transform_rank'] == 4
assert summary['m2']['rho'] == 0.05
assert summary['m2']['explicit_item_features'] is False
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_conditional_id_transform_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)